In [37]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import psycopg2

In [38]:


def get_ticket_price(city):
    try:
        conn = psycopg2.connect(
            host="ep-long-mud-a1jlgayr-pooler.ap-southeast-1.aws.neon.tech",
            port="5432",
            database="airline_data",
            user="neondb_owner",
            password="npg_ra4IbVyH5NMX"
        )

        cur = conn.cursor()
        print("Connected to PostgreSQL!")

        query = """
            SELECT city_name, airfare_usd
            FROM city_airfare
            WHERE city_name ILIKE %s;
        """
        cur.execute(query, (city,))

        results = cur.fetchall()
        if results:
            for city_name, fare in results:
                print(f"City: {city_name}, Airfare: ${fare}")
            return f"Ticket price to {city_name} is ${fare}"
        else:
            print("No matching city found.")
            return "No price data available for this city"

    except Exception as e:
        print("Error:", e)
        return "No price data available for this city"

    finally:
        if 'cur' in locals():
            cur.close()
        if 'conn' in locals():
            conn.close()


In [39]:
OLLAMA_BASE_URL = "http://localhost:11434/v1"
model = "llama3.2"
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')

In [40]:
price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}
tools = [{"type": "function", "function": price_function}]

In [41]:
system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
"""

In [42]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = ollama.chat.completions.create(model=model, messages=messages, tools=tools)
    print(response)
    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = ollama.chat.completions.create(model=model, messages=messages, tools=tools)
    
    return response.choices[0].message.content

In [43]:
def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            if not city or str(city).lower() == "none":
                print("Skipping tool call: destination_city is None or 'none'")
                continue
            price_details = get_ticket_price(city)
            print(price_details)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
    return responses

In [44]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7868
* To create a public link, set `share=True` in `launch()`.


ChatCompletion(id='chatcmpl-279', choices=[Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content='', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_3gskvgrc', function=Function(arguments='{"destination_city":""}', name='get_ticket_price'), type='function', index=0)]))], created=1761583822, model='llama3.2', object='chat.completion', service_tier=None, system_fingerprint='fp_ollama', usage=CompletionUsage(completion_tokens=15, prompt_tokens=211, total_tokens=226, completion_tokens_details=None, prompt_tokens_details=None))
Skipping tool call: destination_city is None or 'none'
ChatCompletion(id='chatcmpl-823', choices=[Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content='', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='